In [1]:
!pip install langchain langchain-text-splitters langchain-community bs4
!pip install -U langchain-mistralai


[notice] A new release of pip is available: 23.2.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
  Obtaining dependency information for langchain-mistralai from https://files.pythonhosted.org/packages/e6/5e/d1ed75e38ce431c49b609a524fc178ecbc38cc0a3ba44b5c3b935d82d824/langchain_mistralai-1.0.1-py3-none-any.whl.metadata
  Obtaining dependency information for tokenizers<1.0.0,>=0.15.1 from https://files.pythonhosted.org/packages/d0/c6/dc3a0db5a6766416c32c034286d7c2d406da1f498e4de04ab1b8959edd00/tokenizers-0.22.1-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata
  Obtaining dependency information for huggingface-hub<2.0,>=0.16.4 from https://files.pythonhosted.org/packages/33/21/e15d90fd09b56938502a0348d566f1915f9789c5bb6c00c1402dc7259b6e/huggingface_hub-1.1.2-py3-none-any.whl.metadata
  Obtaining dependency information for filelock from https://files.pythonhosted.org/packages/76/91/7216b27286936c16f5b4d0c530087e4a54eead683e6b0b73dd0c64844af6/filelock-3.20.0-p

### Habilitando o LangSmith (Opcional)

O Langsmith é uma ferramenta que permite a observabilidade nos nossos sistemas de agentes

- Crie uma conta no langsmith: https://smith.langchain.com/
- Na página principal, após o login, clique em "settings". Depois "API Keys"
- Por fim clique em "+ API Key" e crie uma nova chave do tipo "Personal Access Token"

In [3]:
import os
from dotenv import load_dotenv
load_dotenv("/workspaces/ml-supervised-dev/.env")

langsmith_key = os.getenv("LANGSMITH_API_KEY")
if not langsmith_key:
    print("API KEY não fornecida")

### Escolhendo o modelo de chat

Estamos usando o Mistral como provedor e o modelo "mistral-small-latest".

É necessário ter uma variável de ambiente (MISTRAL_API_KEY) com uma API Key configurada

In [4]:
'''
Cria objeto para consumir modelos do Mistral. Outros provedores são bem parecidos.
Link da documentação: https://python.langchain.com/api_reference/mistralai/chat_models/langchain_mistralai.chat_models.ChatMistralAI.html#langchain_mistralai.chat_models.ChatMistralAI.get_num_tokens_from_messages
'''

import os
from langchain.chat_models import init_chat_model

model = init_chat_model("mistral-small-latest")

In [5]:
model

ChatMistralAI(client=<httpx.Client object at 0x79ead464b770>, async_client=<httpx.AsyncClient object at 0x79ead4f22ab0>, endpoint='https://api.mistral.ai/v1', model='mistral-small-latest', model_kwargs={})

### Definindo o modelo de embeddings

Aqui escolhemos o modelo que será usado para "vetorizar" a query do usuário e as informações em nossa base de dados

In [6]:
from langchain_mistralai import MistralAIEmbeddings

embeddings = MistralAIEmbeddings(model="mistral-embed")

/workspaces/ml-supervised-dev/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
embeddings

MistralAIEmbeddings(client=<httpx.Client object at 0x79ead5738980>, async_client=<httpx.AsyncClient object at 0x79ead57118b0>, mistral_api_key=SecretStr(''), endpoint='https://api.mistral.ai/v1/', max_retries=5, timeout=120, wait_time=30, max_concurrent_requests=64, tokenizer=Tokenizer(version="1.0", truncation=None, padding=None, added_tokens=[{"id":0, "content":"<unk>", "single_word":False, "lstrip":False, "rstrip":False, "normalized":False, "special":True}, {"id":1, "content":"<s>", "single_word":False, "lstrip":False, "rstrip":False, "normalized":False, "special":True}, {"id":2, "content":"</s>", "single_word":False, "lstrip":False, "rstrip":False, "normalized":False, "special":True}], normalizer=Sequence(normalizers=[Prepend(prepend="▁"), Replace(pattern=String(" "), content="▁")]), pre_tokenizer=None, post_processor=TemplateProcessing(single=[SpecialToken(id="<s>", type_id=0), Sequence(id=A, type_id=0)], pair=[SpecialToken(id="<s>", type_id=0), Sequence(id=A, type_id=0), SpecialT

### Definindo o método de armazenamento de vetores (Vector Store)

Neste exemplo, por motivos de simplicidade, vamos salvar os vetores em memória.

In [8]:
!pip install -U "langchain-core"


[notice] A new release of pip is available: 23.2.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [1]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)

NameError: name 'embeddings' is not defined

### Indexing

Nesta etapa buscamos as informações complementares para o nosso modelo

In [3]:
'''
Documento loaders. Objetos que nos permitem acessar
e carregas os "documents" onde estão as informações que queremos
'''

import bs4 # Ferramenta para scraping de páginas web.
from langchain_community.document_loaders import WebBaseLoader # Carrega documentos de páginas web.

# Escolhemos apenas algumas classes CSS para evitar carregar menus, rodapés, etc.
bs4_strainer = bs4.SoupStrainer(class_=("p", "h2"))
loader = WebBaseLoader(
    web_paths=("https://microservices.io/patterns/data/database-per-service.html","https://microservices.io/patterns/reliability/circuit-breaker.html",),
    bs_kwargs={"parse_only": bs4_strainer},
)
docs = loader.load()

assert len(docs) == 2
print(f"Total characters: {len(docs[0].page_content)}")

Total characters: 0


In [4]:
print(docs[0].page_content[:500])

### Spliting

O texto completo pode ser maior que o contexto do nosso modelo. Por esse motivo é necessário dividir em "chunks" para gerar os vetores

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # chunk size (characters)
    chunk_overlap=200,  # chunk overlap (characters)
    add_start_index=True,  # track index in original document
)
all_splits = text_splitter.split_documents(docs)

print(f"Split blog post into {len(all_splits)} sub-documents.")

Split blog post into 0 sub-documents.


In [6]:
all_splits[0]

IndexError: list index out of range

### Storing

Armazenamos os chunks em memória, já convertidos em vetores. Idealmente **não salvamos em memória mas sim em bancos de dados vetoriais**

In [ ]:
document_ids = vector_store.add_documents(documents=all_splits)

print(document_ids[:3])

### Retrieval

Criamos uma tool do tipo code-agent que irá executar o processo de recuperação da informação em nossa base vetorial

In [18]:
from langchain.tools import tool

@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """Retrieve information to help answer a query."""
    retrieved_docs = vector_store.similarity_search(query, k=2)
    serialized = "\n\n".join(
        (f"Source: {doc.metadata}\nContent: {doc.page_content}")
        for doc in retrieved_docs
    )
    return serialized, retrieved_docs

### Agente

In [ ]:
from langchain.agents import create_agent


tools = [retrieve_context]

prompt = (
    "You have access to a tool that retrieves context from a blog post. "
    "Use the tool to help answer user queries."
)
agent = create_agent(model, tools, system_prompt=prompt)

In [ ]:
query = (
    "What is the standard method for Task Decomposition?\n\n"
    "Once you get the answer, look up common extensions of that method."
)

for event in agent.stream( # Streaming response. Permite ver a resposta parcial enquanto é gerada
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
):
    event["messages"][-1].pretty_print()

================================ Human Message =================================

What is the standard method for Task Decomposition?

Once you get the answer, look up common extensions of that method.
================================== Ai Message ==================================
Tool Calls:
  retrieve_context (oIM0C7f2Q)
 Call ID: oIM0C7f2Q
  Args:
    query: standard method for task decomposition
================================= Tool Message =================================
Name: retrieve_context

Source: {'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'start_index': 2578}
Content: Task decomposition can be done (1) by LLM with simple prompting like "Steps for XYZ.\n1.", "What are the subgoals for achieving XYZ?", (2) by using task-specific instructions; e.g. "Write a story outline." for writing a novel, or (3) with human inputs.
Another quite distinct approach, LLM+P (Liu et al. 2023), involves relying on an external classical planner to do long-horizon planni